# Оценка ТЗ по меню виртуального ассистента

Запустите ячейки по порядку. Укажите `TZ_PATH` в папке `TZ` и при необходимости поменяйте модель.

Ключ `OPENAI_API_KEY` читается из `.env` в корне репозитория или `evaluator/.env` (приоритет у последнего).
Текст ТЗ передаётся в OpenAI API. Поддерживаются DOCX (включая таблицы), текстовые PDF, TXT и MD.
Сканы требуют предварительного OCR; рисунки и вложенные объекты DOCX автоматически не распознаются.

«Успешно проставлено» означает наличие одной из четырёх оценок, а не независимую проверку её правильности.
Уверенность — самооценка модели. Неясные требования помечаются «не определено».
Результаты требуют проверки аналитиком. Повторный запуск создаёт отдельную папку результатов.

In [ ]:
from pathlib import Path
_install_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                     if (p / "evaluator/requirements.txt").is_file())
_requirements = str(_install_root / "evaluator/requirements.txt")
%pip install -r "{_requirements}"

Зависимости устанавливаются в текущее ядро. После первой установки при необходимости перезапустите ядро.

In [1]:
from pathlib import Path
import sys
import os
from datetime import datetime, timezone
from uuid import uuid4

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "evaluator/effort_evaluator.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Откройте ноутбук из корня репозитория или папки evalutator.")
sys.path.insert(0, str(ROOT / "evaluator"))
from effort_evaluator import load_menu, load_spec, evaluate, summarize, statistics, save_results, save_summary
from dotenv import dotenv_values
from openai import OpenAI
from IPython.display import display

# Настройки пользователя
MENU_PATH = ROOT / "evaluator/chatbot_virtual_assistant_effort_menu.csv"
TZ_DIR = ROOT / "evaluator/TZ"
TZ_PATH = TZ_DIR / "ТЗ - Сервис для создания виртуальных ассистентов (Electra 2.0).docx"
BATCH_SIZE = 7
MAX_SPEC_CHARS = 180_000  # Защита от переполнения контекста; ТЗ не обрезается.

config = {**dotenv_values(ROOT / ".env"), **dotenv_values(ROOT / "evaluator/.env")}
MODEL = config.get("OPENAI_EVALUATOR_MODEL") or config.get("OPENAI_LLM_MODEL") or "gpt-4.1-mini"
RUN_DIR = ROOT / "evaluator/results" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8])

In [2]:
menu = load_menu(MENU_PATH)
text = load_spec(TZ_PATH, TZ_DIR)
print(f"ТЗ: {TZ_PATH.name}; символов: {len(text):,}; пунктов меню: {len(menu)}; модель: {MODEL}")
display(menu)
if len(text) > MAX_SPEC_CHARS:
    raise ValueError("ТЗ превышает MAX_SPEC_CHARS. Увеличьте лимит с учётом контекста модели.")

ТЗ: ТЗ - Сервис для создания виртуальных ассистентов (Electra 2.0).docx; символов: 36,361; пунктов меню: 42; модель: gpt-4.1-mini


,Блок,Компонент,Отсутствует,Small,Medium,Large
0,Каналы и интерфейсы,Текстовый веб-чат,Не нужен.,Готовый web-виджет или простая страница чата; ...,Кастомный UI в существующем продукте; авториза...,Полноценный омниканальный интерфейс с кастомны...
1,Каналы и интерфейсы,Мессенджеры и внешние каналы,Не нужны.,Один стандартный канал: Telegram / Slack / Tea...,"2–4 канала, единая логика сессий и пользовател...","Много каналов, contact-center/CRM, сложная мар..."
2,Каналы и интерфейсы,Мобильное приложение / SDK,Не требуется.,Интеграция в существующее приложение через Web...,"Нативная интеграция iOS/Android, push-уведомле...","Отдельный мобильный UX, offline/poor-network с..."
3,Голос,Голосовой ввод (Speech-to-Text),Не нужен.,"Стандартный SaaS STT (например, Whisper API / ...",On-prem / private cloud STT либо повышенные тр...,On-prem с высокими SLA и масштабом; специфичес...
4,Голос,Голосовой вывод (Text-to-Speech),Не нужен.,Стандартный SaaS TTS с готовым голосом.,On-prem/private deployment или кастомный голос...,Собственный/брендовый голос с датасетом и дооб...
5,Голос,Телефонный канал / телефония,Не нужен.,Подключение одного SIP/телефония-провайдера; в...,Интеграция с корпоративной телефонией/contact ...,Высоконагруженный voice-bot: тысячи параллельн...
6,LLM и диалог,Базовая LLM / генерация ответов,LLM не используется; ответы по фиксированным п...,"Одна облачная LLM через API, базовый system pr...","Несколько моделей/провайдеров, fallback, маршр...",On-prem/self-hosted LLM или сложный model gate...
7,LLM и диалог,Промптинг и управление поведением,Не требуется.,Один основной системный промпт и несколько про...,"Набор сценариев и ролей, structured outputs, p...","Большая prompt/agent policy-система, динамичес..."
8,LLM и диалог,Управление контекстом диалога,Каждое сообщение независимо.,Хранение последних N сообщений одной сессии.,"Суммаризация длинных диалогов, выделение факто...",Долгосрочная память по пользователю/организаци...
9,LLM и диалог,Сценарии / state machine,Свободный чат без фиксированных сценариев.,Несколько простых intent → response/action сце...,"Многошаговые бизнес-процессы с состояниями, ва...","Сложные динамические workflow, десятки процесс..."


In [3]:
api_key = config.get("OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Добавьте OPENAI_API_KEY в .env в корне репозитория или evalutator/.env.")
client = OpenAI(api_key=api_key, timeout=180.0, max_retries=3)
results = evaluate(client, MODEL, menu, text, batch_size=BATCH_SIZE, max_spec_chars=MAX_SPEC_CHARS)
# Сохраняем оценки ДО запроса заключения: ошибка второго этапа не потеряет разметку.
csv_path = save_results(results, RUN_DIR, TZ_PATH.stem)
print(f"Разметка сохранена: {csv_path}")
display(results)
display(__import__("pandas").DataFrame([statistics(results)]))

Размечено 7 из 42 пунктов
Размечено 14 из 42 пунктов
Размечено 21 из 42 пунктов
Размечено 28 из 42 пунктов
Размечено 35 из 42 пунктов
Размечено 42 из 42 пунктов
Разметка сохранена: /home/elwis/Projects/case-finder/evaluator/results/20261028T165535Z_f5dd596b/ТЗ - Сервис для создания виртуальных ассистентов (Electra 2.0)_оценка.csv


,Блок,Компонент,Оценка,Уверенность,Коментарий
0,Каналы и интерфейсы,Текстовый веб-чат,medium,0.90,По пункту 1.2 описана работа текстового веб-ча...
1,Каналы и интерфейсы,Мессенджеры и внешние каналы,small,0.95,"Пункт 7.3 указывает, что интеграции с мессендж..."
2,Каналы и интерфейсы,Мобильное приложение / SDK,отсутствует,0.95,В описании нет упоминаний или требований по мо...
3,Голос,Голосовой ввод (Speech-to-Text),отсутствует,0.95,"В разделе 1.3 указано, что поддержка голосовых..."
4,Голос,Голосовой вывод (Text-to-Speech),отсутствует,0.95,"Пункт 1.3 прямо говорит, что голосовые ответы ..."
5,Голос,Телефонный канал / телефония,отсутствует,1.00,В тексте нет упоминаний о телефонном канале ил...
6,LLM и диалог,Базовая LLM / генерация ответов,small,0.90,В ТЗ не указано использование LLM для генераци...
7,LLM и диалог,Промптинг и управление поведением,small,0.90,В ТЗ описаана система на On-Prem и Cloud с баз...
8,LLM и диалог,Управление контекстом диалога,medium,0.85,"В ТЗ указано, что диалоги имеют контекст: хран..."
9,LLM и диалог,Сценарии / state machine,medium,0.90,"ТЗ описывает многошаговые сценарии, состояния ..."


,Всего,Успешно проставлено,Не определены,отсутствует,small,medium,large
0,42,42,0,4,12,25,1


In [4]:
summary = summarize(client, MODEL, menu, text, results)
additions, report_path = save_summary(summary, results, RUN_DIR, TZ_PATH.stem)
print("Общее заключение\n")
print(summary.conclusion)
print("\nВопросы для уточнения:")
for question in summary.questions:
    print("•", question)
print("\nРекомендуемые дополнения меню с критериями по категориям:")
if additions.empty:
    print("Дополнения не требуются.")
else:
    with __import__("pandas").option_context("display.max_colwidth", None):
        display(additions)
print(f"\nЗаключение: {report_path}")

Общее заключение

Данное ТЗ описывает комплексную систему для создания виртуальных ассистентов с функционалом On-Prem/Cloud, поддержкой ролей, управления чатботами и их контентом. Многие аспекты покрыты на среднем уровне, например интеграции, сценарии, аналитика, безопасность и инфраструктура. В тоже время мультиязычность и конфигурируемость хорошо проработаны, обеспечивая высокий уровень удобства. Голосовые каналы, мобильные приложения, полноценные LLM, телефонный канал и продвинутая автоматизация CI/CD отсутствуют или представлены на базовом уровне.

Вопросы для уточнения:
• Подтверждается ли необходимость поддержки голосового ввода (speech-to-text) и голосового ответа (text-to-speech) в ближайших этапах?
• Требуется ли интеграция с корпоративными IAM через SSO/OIDC/SAML?
• Нужна ли полноценная автоматизация CI/CD с управлением версиями и откатами на уровне системы, а не вручную?
• Требуется ли поддержка масштабных голосовых звонков и интеграция с контакт-центром (телефония)?
• Какой

,Блок,Компонент,Обоснование,Отсутствует,Small,Medium,Large
0,Голос,Голосовой ввод (Speech-to-Text),"В ТЗ голосовой ввод отсутствует, но может потребоваться для поддержки голосового общения с чатботом.","Голосовой ввод не реализован, голосовые запросы не поддерживаются.","Интеграция стандартного SaaS STT сервиса без обучения, несколько языков.","On-prem/Private Cloud с повышенными требованиями latency, словарём терминов.","Сложный On-prem движок с высокими SLA, адаптацией, потоковой обработкой и многоканальным diarization."
1,Голос,Голосовой вывод (Text-to-Speech),"В ТЗ голосовой вывод отсутствует, но клиент должен получить голосовой ответ в будущем.",Голосовой вывод не поддерживается.,Использование стандартного SaaS TTS с готовыми голосами.,On-prem или кастомный голос с voice cloning и streaming.,"Брендовый голос с эмоциями, многъязычностью и низкой задержкой."
2,Каналы и интерфейсы,Мобильное приложение / SDK,"В ТЗ отсутствует упоминание мобильного приложения или SDK, но такие интеграции могут понадобиться для удобства пользователей.",Отсутствует поддержка мобильных приложений и SDK.,Интеграция через WebView или простой SDK.,Нативные мобильные приложения с push-уведомлениями и микрофоном.,Отдельный мобильный UX с offline поддержкой и собственным SDK.



Заключение: /home/elwis/Projects/case-finder/evaluator/results/20261028T165535Z_f5dd596b/ТЗ - Сервис для создания виртуальных ассистентов (Electra 2.0)_заключение.txt


Файлы сохраняются в `evalutator/results/<идентификатор запуска>/`:
- `*_оценка.csv`: **Блок, Компонент, Оценка, Уверенность, Коментарий** (написание последнего столбца сохранено по запросу).
- `*_заключение.txt`: заключение, статистика, вопросы и расшифровка дополнений.
- `*_дополнения_меню.csv`: новые компоненты с определениями отсутствует/small/medium/large.

CSV имеет кодировку UTF-8 с BOM для Excel. Результаты и `.env` исключены из Git.
При отказе API, ошибке структуры ответа, пропусках или повторах ID выполнение останавливается с ошибкой;
техническая ошибка не превращается в оценку «не определено». После ошибки заключения можно повторить только последнюю кодовую ячейку.

In [ ]:
!pip install -U openai